In [3]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy.spatial.distance import euclidean

In [8]:
# configuration
SEED = 82
DISTANCE_METRIC = "euclidean"

PATHS = {
    "original": "../../dataset/X_scaled.csv",
    "frozen_dec": "../../final_dec/model/final_embeddings.npy",
    "idec": "../../final_idec/model/final_embeddings.npy",
    "trainable_dec": "../encoder_ablation/runs/trainable/Z_full.npy",
}

OUTPUT_DIR = Path("experiments/representation_preservation/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# load data
X_original = pd.read_csv(PATHS["original"]).to_numpy(dtype=np.float64)
Z_frozen = np.load(PATHS["frozen_dec"], allow_pickle=False)
Z_trainable = np.load(PATHS["trainable_dec"], allow_pickle=False)
Z_idec = np.load(PATHS["idec"], allow_pickle=False)

embeddings = {
    "Frozen DEC": Z_frozen,
    "Trainable DEC": Z_trainable,
    "IDEC": Z_idec,
}

In [ ]:
# Validate shapes and finite values

N = X_original.shape[0]

assert X_original.shape == (N, 7), (
    f"Expected 7 original features. Got {X_original.shape}"
)

for name, Z in embeddings.items():
  assert Z.shape == (N, 5), (
      f"{name}: expected shape ({N}, 5). Got {Z.shape}"
  )
  assert np.isfinite(Z).all(), f"{name} contains NaN or infinity"

assert np.isfinite(X_original).all(), (
    "Original features contain NaN or infinity"
)

print("All arrays loaded and passed shape/finite-value checks.")
print(f"Observations: {N:,}")
print(f"Original space: {X_original.shape}")
for name, Z in embeddings.items():
    print(f"{name}: {Z.shape}")

All arrays loaded and passed shape/finite-value checks.
Observations: 99,997
Original space: (99997, 7)
Frozen DEC: (99997, 5)
Trainable DEC: (99997, 5)
IDEC: (99997, 5)


In [ ]:
# Define the evaluation protocol
protocol = {
    "experiment": "Representation Preservation",
    "observations": int(N),
    "sampling": "None. full dataset",
    "random_seed": SEED,
    "original_feature_dimensions": int(X_original.shape[1]),
    "latent_dimensions": {
        name: int(Z.shape[1])
        for name, Z in embeddings.items()
    },
    "distance_metric": DISTANCE_METRIC,
    "models": list(embeddings.keys()),
}

print("\nEvaluation protocol:")
print(json.dumps(protocol, indent=2))

with open(OUTPUT_DIR / "representation_preservation_protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

print("\nProtocol saved to representation_preservation_protocol.json")



Evaluation protocol:
{
  "experiment": "Representation Preservation",
  "observations": 99997,
  "sampling": "None. full dataset",
  "random_seed": 82,
  "original_feature_dimensions": 7,
  "latent_dimensions": {
    "Frozen DEC": 5,
    "Trainable DEC": 5,
    "IDEC": 5
  },
  "distance_metric": "euclidean",
  "models": [
    "Frozen DEC",
    "Trainable DEC",
    "IDEC"
  ]
}

Protocol saved to representation_preservation_protocol.json


# KNN preservation

In [ ]:
from sklearn.neighbors import NearestNeighbors
import time

K_VALUES = [10, 50]
MAX_K = max(K_VALUES)

def find_knn(data, k=MAX_K):
  """
  Find the k nearest neighbors of every observation.
  """
  nn = NearestNeighbors(
      n_neighbors=k,
      metric=DISTANCE_METRIC,
      algorithm="auto",
      n_jobs=-1,
  )

  nn.fit(data)

  neighbors = nn.kneighbors(
      X=None,
      return_distance=False,
  )

  # store as int32 to reduce memory usage
  return neighbors.astype(np.int32)

# find neighbors in the original 7D space
start = time.time()

print("Finding neighbors in the original feature space...")
original_neighbors = find_knn(X_original)

print(f"Original space completed in {(time.time() - start) / 60:.2f} minutes.")
print("Neighbor-index array shape:", original_neighbors.shape)

Finding neighbors in the original feature space...
Original space completed in 0.60 minutes.
Neighbor-index array shape: (99997, 50)


In [ ]:
latent_neighbors = {}

for name, Z in embeddings.items():
    start = time.time()

    print(f"Finding neighbors for {name}...")
    latent_neighbors[name] = find_knn(Z)

    elapsed = (time.time() - start) / 60
    print(f"Completed in {elapsed:.2f} minutes.")
    print("Neighbor-index array shape:", latent_neighbors[name].shape)
    print()

Finding neighbors for Frozen DEC...
Completed in 0.13 minutes.
Neighbor-index array shape: (99997, 50)

Finding neighbors for Trainable DEC...
Completed in 0.08 minutes.
Neighbor-index array shape: (99997, 50)

Finding neighbors for IDEC...
Completed in 0.10 minutes.
Neighbor-index array shape: (99997, 50)



In [ ]:
def calculate_knn_preservation(
    original_neighbors,
    latent_neighbors,
    k,
    block_size=2000
):
    """
    Calculate per-observation KNN preservation.

    For each observation:
        preservation = shared neighbors / k

    Processing in blocks limits memory usage.
    """
    n = original_neighbors.shape[0]
    scores = np.empty(n, dtype=np.float32)

    for start in range(0, n, block_size):
        end = min(start + block_size, n)

        original_block = original_neighbors[start:end, :k]
        latent_block = latent_neighbors[start:end, :k]

        # Compare each observation's k neighbors across the two spaces.
        # Shape: (block_size, k, k)
        matches = (
            original_block[:, :, None]
            == latent_block[:, None, :]
        )

        # Count shared neighbors for each observation
        shared_counts = matches.any(axis=2).sum(axis=1)

        # Convert counts into scores between 0 and 1
        scores[start:end] = shared_counts / k

    return scores

In [ ]:
results = []
preservation_scores = {}

for name, model_neighbors in latent_neighbors.items():
    for k in K_VALUES:

        scores = calculate_knn_preservation(
            original_neighbors=original_neighbors,
            latent_neighbors=model_neighbors,
            k=k
        )

        preservation_scores[(name, k)] = scores

        results.append({
            "Model": name,
            "k": k,
            "Mean preservation": scores.mean(),
            "Median preservation": np.median(scores),
            "Std preservation": scores.std()
        })

results_df = pd.DataFrame(results)

print("KNN Preservation Results")
display(
    results_df.style.format({
        "Mean preservation": "{:.4f}",
        "Median preservation": "{:.4f}",
        "Std preservation": "{:.4f}"
    })
)

results_df.to_csv(
    OUTPUT_DIR / "knn_preservation_results.csv",
    index=False
)

print("\nResults saved to knn_preservation_results.csv")

KNN Preservation Results


,Model,k,Mean preservation,Median preservation,Std preservation
0,Frozen DEC,10,0.1337,0.1000,0.1199
1,Frozen DEC,50,0.1809,0.1800,0.0993
2,Trainable DEC,10,0.0179,0.0000,0.0483
3,Trainable DEC,50,0.0322,0.0200,0.0650
4,IDEC,10,0.0873,0.1000,0.0981
5,IDEC,50,0.1130,0.1000,0.0837



Results saved to knn_preservation_results.csv


KNN preservation was evaluated for all 99,997 observations using Euclidean distance, with neighborhood sizes (k=10) and (k=50). The measure represents the proportion of each observation's original-space nearest neighbors that remain among its nearest neighbors in the corresponding latent space.

***Three important observations***
1. **Frozen DEC preserves the most local structure.**
It retains substantially more of the original nearest-neighbor relationships than either trainable model. This is consistent with the idea that fine-tuning the encoder changes the representation.

2. **Trainable DEC shows particularly substantial changes.**
Its mean preservation is only 1.79% at k=10 and 3.22% at k=50. This suggests that its latent neighborhoods differ noticeably from those in the original feature space.
However, low preservation does not automatically mean the representation is poor. DEC optimizes a clustering objective, not an objective designed to preserve all original nearest neighbors.

3. **IDEC appears to moderate this loss of neighborhood preservation.**
IDEC performs better than Trainable DEC at both neighborhood sizes, but worse than Frozen DEC. This pattern is consistent with reconstruction regularization helping retain some original local structure.


# Pairwise-distance correlation

In [ ]:
def exact_multi_embedding_distance_correlations(
    X_original,
    embeddings,
    block_size=200
):
    """
    Calculate exact pairwise-distance Pearson correlations between
    the original space and multiple embeddings in a single pass.

    Each unique observation pair is processed exactly once.
    Original-space distances are calculated once and reused.
    """

    X_original = np.asarray(X_original, dtype=np.float64)

    if X_original.ndim != 2:
        raise ValueError("X_original must be a 2D array.")

    if not np.isfinite(X_original).all():
        raise ValueError("X_original contains NaN or infinity.")

    n_observations = X_original.shape[0]

    # Validate embeddings
    validated_embeddings = {}

    for name, Z in embeddings.items():
        Z = np.asarray(Z, dtype=np.float64)

        if Z.ndim != 2 or Z.shape[0] != n_observations:
            raise ValueError(
                f"{name}: embedding must be 2D and have "
                f"{n_observations} observations."
            )

        if not np.isfinite(Z).all():
            raise ValueError(f"{name} contains NaN or infinity.")

        validated_embeddings[name] = Z

    # Running statistics for each embedding
    stats = {
        name: {
            "n": 0,
            "sum_x": 0.0,
            "sum_y": 0.0,
            "sum_x2": 0.0,
            "sum_y2": 0.0,
            "sum_xy": 0.0,
        }
        for name in validated_embeddings
    }

    for start in range(0, n_observations, block_size):
        end = min(start + block_size, n_observations)

        X_block = X_original[start:end]

        # Calculate original-space distances to later observations.
        # Reuse these distances for every embedding.
        if end < n_observations:
            D_X_tail = cdist(
                X_block,
                X_original[end:],
                metric="euclidean"
            )
        else:
            D_X_tail = None

        # Calculate original-space distances within this block.
        D_X_block = cdist(
            X_block,
            X_block,
            metric="euclidean"
        )

        rows, cols = np.triu_indices(end - start, k=1)
        x_within = D_X_block[rows, cols]

        if D_X_tail is not None:
            x_tail = D_X_tail.ravel()

        for name, Z in validated_embeddings.items():
            Z_block = Z[start:end]

            # Distances within the current block
            D_Z_block = cdist(
                Z_block,
                Z_block,
                metric="euclidean"
            )
            y_within = D_Z_block[rows, cols]

            s = stats[name]

            s["n"] += x_within.size
            s["sum_x"] += x_within.sum()
            s["sum_y"] += y_within.sum()
            s["sum_x2"] += np.dot(x_within, x_within)
            s["sum_y2"] += np.dot(y_within, y_within)
            s["sum_xy"] += np.dot(x_within, y_within)

            if D_X_tail is not None:
                D_Z_tail = cdist(
                    Z_block,
                    Z[end:],
                    metric="euclidean"
                )
                y_tail = D_Z_tail.ravel()

                s["n"] += x_tail.size
                s["sum_x"] += x_tail.sum()
                s["sum_y"] += y_tail.sum()
                s["sum_x2"] += np.dot(x_tail, x_tail)
                s["sum_y2"] += np.dot(y_tail, y_tail)
                s["sum_xy"] += np.dot(x_tail, y_tail)

    # Convert running statistics to Pearson correlations
    results = {}

    for name, s in stats.items():
        n = s["n"]

        numerator = n * s["sum_xy"] - s["sum_x"] * s["sum_y"]

        denominator_x = n * s["sum_x2"] - s["sum_x"] ** 2
        denominator_y = n * s["sum_y2"] - s["sum_y"] ** 2

        denominator = np.sqrt(denominator_x * denominator_y)

        if denominator == 0:
            raise ValueError(
                f"{name}: correlation is undefined because "
                "one distance vector has zero variance."
            )

        results[name] = {
            "pearson_correlation": float(numerator / denominator),
            "number_of_pairs": int(n),
        }

    expected_pairs = n_observations * (n_observations - 1) // 2

    for name, result in results.items():
        if result["number_of_pairs"] != expected_pairs:
            raise RuntimeError(
                f"{name}: expected {expected_pairs} pairs, "
                f"got {result['number_of_pairs']}."
            )

    return results

In [ ]:
import time

start_time = time.time()

full_results = exact_multi_embedding_distance_correlations(
    X_original=X_original,
    embeddings=embeddings,
    block_size=200
)

elapsed = time.time() - start_time

print(f"Observations: {len(X_original):,}")
print(
    "Expected unique pairs:",
    len(X_original) * (len(X_original) - 1) // 2
)
print(f"Actual runtime: {elapsed / 60:.2f} minutes")

print("\nFull-dataset pairwise-distance correlations:")
for name, result in full_results.items():
    print(
        f"{name}: r = {result['pearson_correlation']:.6f}, "
        f"pairs = {result['number_of_pairs']:,}"
    )

Observations: 99,997
Expected unique pairs: 4999650006
Actual runtime: 5.04 minutes

Full-dataset pairwise-distance correlations:
Frozen DEC: r = 0.697885, pairs = 4,999,650,006
Trainable DEC: r = 0.366610, pairs = 4,999,650,006
IDEC: r = 0.391808, pairs = 4,999,650,006


In [ ]:
import json
import pandas as pd
from pathlib import Path


# 1. Save correlation results as CSV
results_df = pd.DataFrame([
    {
        "model": name,
        "pearson_distance_correlation": result["pearson_correlation"],
        "number_of_observations": len(X_original),
        "number_of_pairs": result["number_of_pairs"],
    }
    for name, result in full_results.items()
])

csv_path = OUTPUT_DIR / "pairwise_distance_correlations.csv"
results_df.to_csv(csv_path, index=False)

# 2. Save results and experimental metadata as JSON
experiment_record = {
    "experiment": "Representation Preservation",
    "measurement": "Exact all-pairs Euclidean distance correlation",
    "correlation_method": "Pearson",
    "number_of_observations": int(len(X_original)),
    "original_space_dimensions": int(X_original.shape[1]),
    "latent_space_dimensions": {
        name: int(Z.shape[1])
        for name, Z in embeddings.items()
    },
    "number_of_unique_pairs": int(
        len(X_original) * (len(X_original) - 1) // 2
    ),
    "distance_metric": "euclidean",
    "pair_sampling": False,
    "block_size": 200,
    "runtime_seconds": round(elapsed, 3),
    "random_seed": SEED,
    "results": {
        name: {
            "pearson_distance_correlation": result["pearson_correlation"],
            "number_of_pairs": result["number_of_pairs"],
        }
        for name, result in full_results.items()
    },
}

json_path = OUTPUT_DIR / "pairwise_distance_correlations.json"

with open(json_path, "w") as f:
    json.dump(experiment_record, f, indent=2)

print("Saved files:")
print(csv_path)
print(json_path)

print("\nCSV preview:")
display(results_df)

Saved files:
/content/representation_preservation_results/pairwise_distance_correlations.csv
/content/representation_preservation_results/pairwise_distance_correlations.json

CSV preview:


,model,pearson_distance_correlation,number_of_observations,number_of_pairs
0,Frozen DEC,0.697885,99997,4999650006
1,Trainable DEC,0.366610,99997,4999650006
2,IDEC,0.391808,99997,4999650006


Pairwise-distance preservation was assessed by calculating the Pearson correlation between Euclidean distances in the original 7-dimensional feature space and the corresponding distances in each 5-dimensional latent space. The analysis included all 99,997 observations, covering 4,999,650,006 unique observation pairs per model. Exact correlations were computed using a blockwise procedure without sampling pairs.

***Three important observations***

1. **Frozen DEC retains the strongest distance relationships.**  
   It achieved the highest Pearson correlation (r = 0.6979), indicating the strongest linear association between original-space and latent-space pairwise distances among the three models.

2. **Trainable DEC shows the weakest distance association.**  
   Its correlation was substantially lower (r = 0.3666), indicating that pairwise distances in its latent space have a weaker linear relationship with those in the original feature space.

3. **IDEC shows a modest improvement over Trainable DEC.**  
   IDEC achieved a correlation of (r = 0.3918), slightly higher than Trainable DEC but considerably lower than Frozen DEC. This pattern is consistent with IDEC retaining somewhat more of the original distance relationships than Trainable DEC.

**Combined interpretation of local and global structure**

The KNN preservation and pairwise-distance correlation results show a consistent pattern across the three models: Frozen DEC preserves the most structure from the original feature space, IDEC is intermediate, and Trainable DEC preserves the least according to both measures.

Frozen DEC achieved the highest KNN preservation (13.37% at (k=10); 18.09% at (k=50)) and the highest pairwise-distance correlation (r=0.6979). Trainable DEC showed the lowest KNN preservation (1.79% at (k=10); 3.22% at (k=50)) and the lowest distance correlation (r=0.3666). IDEC performed between the two models on both measures, with KNN preservation of 8.73% at (k=10) and 11.30% at (k=50), and a distance correlation of (r=0.3918).

Together, these findings indicate that the trainable latent representations differ substantially from the original feature space in both local neighborhood composition and pairwise-distance relationships. IDEC retains more of these properties than Trainable DEC, but less than Frozen DEC. This pattern is consistent with reconstruction regularization helping retain some original-space structure.

It should be noted that DEC is designed to optimize a clustering objective rather than preserve all original distances or nearest neighbors, and these results characterize representation changes.